In [ ]:
# Python script for cleaning UAV orthomoasaic from Point Blue into polygon area
# STEP 3 vectorise
# Copyright (C) 2025 Alexandra Strang  and Gorden Jiang

import rioxarray
import geopandas as gpd
from geocube.vector import vectorize
import numpy as np

In [67]:
# --- Configuration ---
# Path to your small, reclassified binary GeoTIFF (e.g., the 42KB file)
INPUT_RASTER_PATH = "reclassified_raster.tif"
# Path for the final vector output file (GeoPackage is recommended)
OUTPUT_VECTOR_PATH = "vectorized_features.gpkg"


In [ ]:
# --- Start of the Vectorization Workflow ---

print(f"Loading raster '{INPUT_RASTER_PATH}' into memory...")

# 1. Open the raster. Because it's small, it will be loaded directly into memory.
#    The .squeeze() method removes the band dimension (e.g., from [1, y, x] to [y, x]),
#    which is what geocube's vectorize function expects.
try:
    binary_raster = rioxarray.open_rasterio(INPUT_RASTER_PATH).squeeze()
except Exception as e:
    print(f"Error: Could not open raster file. {e}")
    exit()

print("Raster loaded successfully.")
print(f"Raster dimensions (y, x): {binary_raster.shape}")
print(f"Data type: {binary_raster.dtype}")

# Ensure the raster is an integer type, which is required for vectorization.
if not np.issubdtype(binary_raster.dtype, np.integer):
    raise TypeError("Raster data type must be an integer for vectorization.")

# 2. Vectorize the entire raster.
#    geocube will create polygons for all groups of contiguous pixels
#    (both for the 0s and the 1s).
print("\nVectorizing the raster...")
try:
    # This function converts the raster to a GeoDataFrame.
    vectorized_gdf = vectorize(binary_raster)
    print(f"Initial vectorization complete. Found {len(vectorized_gdf)} total polygons (for 0s and 1s).")
except Exception as e:
    print(f"An error occurred during vectorization: {e}")
    exit()

# The resulting GeoDataFrame has a 'value' column with the pixel value (0 or 1).
# You can inspect it with: print(vectorized_gdf.head())


Loading raster 'reclassified_raster.tif' into memory...
Raster loaded successfully.
Raster dimensions (y, x): (199, 208)
Data type: uint8

Vectorizing the raster...
Initial vectorization complete. Found 11 total polygons (for 0s and 1s).


c:\Users\gji19\AppData\Local\Programs\Python\Python311\Lib\site-packages\geocube\vector.py:62: UserWarning: The array has no name. Column name defaults to _data
  warnings.warn("The array has no name. Column name defaults to _data")


In [69]:
print(vectorized_gdf.head())


   _data                                           geometry
0    1.0  POLYGON ((253935.19 -1343454.981, 253935.19 -1...
1    1.0  POLYGON ((253922.751 -1343467.422, 253922.751 ...
2    1.0  POLYGON ((253910.312 -1343604.272, 253910.312 ...
3    1.0  POLYGON ((255378.092 -1343766.003, 255378.092 ...
4    1.0  POLYGON ((254146.649 -1343380.335, 254146.649 ...


In [70]:
from shapely.geometry import Polygon

# Function to remove all holes from a polygon
def remove_holes(poly):
    # If the polygon has interiors (holes), create a new polygon from only the exterior ring
    if poly.interiors:
        return Polygon(list(poly.exterior.coords))
    # Otherwise, return the original polygon
    else:
        return poly

# --- Workflow ---

# 2. Remove all holes from each polygon.
print("Removing holes from polygons...")
# The .apply() method runs our function on every geometry in the GeoDataFrame
vectorized_gdf['geometry'] = vectorized_gdf['geometry'].apply(remove_holes)

# 3. Simplify the geometry to smooth the edges.
simplify_tolerance = 5
print(f"Simplifying geometry with tolerance {simplify_tolerance}...")
vectorized_gdf['geometry'] = vectorized_gdf.geometry.simplify(
    tolerance=simplify_tolerance, preserve_topology=True
)


Removing holes from polygons...
Simplifying geometry with tolerance 5...


In [ ]:
# 3. Filter the GeoDataFrame to keep only polygons where the value is 1.
print("Filtering for features with value = 1...")
polygons_of_ones = vectorized_gdf[vectorized_gdf['_data'] == 1]

if polygons_of_ones.empty:
    print("\nWarning: No polygons were found for the value 1.")
    print("The process will stop as there is nothing to save.")
else:
    print(f"Found {len(polygons_of_ones)} polygons corresponding to the value 1.")

    # 4. Save the filtered GeoDataFrame to a vector file.
    #    You can use "GPKG" for GeoPackage or "ESRI Shapefile" for shapefiles.
    print(f"Saving the final polygons to '{OUTPUT_VECTOR_PATH}'...")
    polygons_of_ones.to_file(OUTPUT_VECTOR_PATH, driver="GPKG")

    print("\n-----------------------------------------")
    print("Vectorization complete!")
    print(f"Final output saved to: {OUTPUT_VECTOR_PATH}")
    print("-----------------------------------------")

Filtering for features with value = 1...
Found 11 polygons corresponding to the value 1.
Saving the final polygons to 'vectorized_features.gpkg'...

-----------------------------------------
Vectorization complete!
Final output saved to: vectorized_features.gpkg
-----------------------------------------
